# harness-agent（OpenHarness）Python SDK 示例

[harness-agent](https://pypi.org/project/harness-agent/) 是开源多模型 **编程智能体**（CLI + SDK），仓库见 [AgentBoardTT/openharness](https://github.com/AgentBoardTT/openharness)。

## 运行前准备

1. **Python 3.12+**（本包不支持3.11及以下；若 Jupyter 内核仍是 3.9/3.11，请先创建3.12 虚拟环境并把该内核选为本笔记本的内核）。
2. 安装依赖：
   ```bash
   pip install -r requirements-harness.txt
   ```
3. 配置模型密钥（任选其一）：
   - 终端执行 `harness connect`，或
   - 设置环境变量：`ANTHROPIC_API_KEY` / `OPENAI_API_KEY` / `GOOGLE_API_KEY` 等（与官方文档一致）。

下面示例使用 **`permission_mode="plan"`**（只读规划模式），尽量避免对仓库写文件；若你要让智能体改代码，可改为 `accept_edits` 或 `bypass`（脚本/CI 常用），并自行承担风险。

In [ ]:
import sys

need = (3, 12)
if sys.version_info < need:
    raise RuntimeError(
        f"harness-agent 需要 Python {need[0]}.{need[1]}+，当前为 {sys.version_info.major}.{sys.version_info.minor}。"
        "请切换 Jupyter 内核到 3.12+ 后再运行。"
    )
print(sys.version)

In [ ]:
import harness

# 在 Jupyter 中可直接顶层 await（IPython 7+）
async def demo_simple():
    """流式消费 run() 产出的消息；先打印原始对象便于观察类型。"""
    async for msg in harness.run(
        "用中文用不超过三句话解释 requirements.txt 的作用；不要修改任何文件，不要执行 shell。",
        permission_mode="plan",
        max_turns=8,
    ):
        print(msg)


await demo_simple()

In [ ]:
import harness


async def demo_match():
    """按官方 README 的 match/case 方式处理常见消息类型（若类型名随版本变化，可退回上一格「打印 msg」）。"""
    async for msg in harness.run(
        "同上：简短中文说明 requirements.txt；不要改文件。",
        permission_mode="plan",
        max_turns=8,
    ):
        match msg:
            case harness.TextMessage(text=t, is_partial=False):
                print(t, end="", flush=True)
            case harness.ToolUse(name=name):
                print(f"\n[tool] {name}")
            case harness.Result(text=t, total_tokens=tok):
                print(f"\n\n---\n完成 total_tokens={tok}\n{t}")
            case _:
                pass


await demo_match()

## 显式指定 provider / model（可选）

密钥仍来自 `~/.harness/config.toml` 或对应环境变量。将 `provider` / `model` 换成你账号下可用的组合（也可用 CLI `harness models list` 查看）。

In [ ]:
import os

import harness

provider = os.environ.get("HARNESS_DEMO_PROVIDER", "openai")
model = os.environ.get("HARNESS_DEMO_MODEL", "gpt-4.1")


async def demo_explicit_model():
    async for msg in harness.run(
        "Say hello in one short English sentence.",
        provider=provider,
        model=model,
        permission_mode="plan",
        max_turns=5,
    ):
        print(msg)


await demo_explicit_model()